In [3]:
import sys
sys.path.append('../../../Share')
import config, baseline, Target, Model

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


from collections import Counter

def balance_data(X, y):
    # Count samples per class
    class_counts = Counter(y)
    min_count = min(class_counts.values())  # target: balance all to minority count

    indices_list = []

    for label in sorted(class_counts.keys()):
        label_indices = np.where(y == label)[0]
        selected_indices = np.random.choice(label_indices, size=min_count, replace=False)
        indices_list.extend(selected_indices)

    # Shuffle all selected indices
    balanced_indices = np.random.permutation(indices_list)

    # Subset the data
    X_balanced = X[balanced_indices]
    y_balanced = y[balanced_indices]

    return X_balanced, y_balanced

In [ ]:
SUBJECT = "Hunmin"
baseline_K_val = [i for i in range(len(config.dataset_sub_H))]

ACC_lst = []
for i, K in enumerate(baseline_K_val[1:-1]):
    trainer = baseline.ModelTrainer(config, subject=f"{SUBJECT}")
    X_train, y_train, _, _ = trainer.return_until_K_data(K, train_ratio=1)
    X_train, y_train = balance_data(X_train, y_train)

    X_test, y_test, _, _ = trainer.return_K_th_data_only(K+1, train_ratio=1)
    X_test, y_test = balance_data(X_test, y_test)

    model = Model.Original_model(X_train.shape[1:], num_class=6)

    #print(pd.Series(y_test).value_counts())

    history, _ = Model.Train_model_without_test(model, X_train, y_train, set_epoch=100, set_batch_size=128, Model_name='', set_verbose=False, save_model_set=False)
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    ACC_lst.append(acc)
    print(ACC_lst, f"{i}/{len(baseline_K_val)}\n\n")
    pd.DataFrame(history.history).to_csv(f'./Result/{SUBJECT}/train_with_{K}_test_with_next_session.csv', index=False)


Dataset 1/66 - Session Exp_2025-05-27/E8331D05289A/
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
Start Training (total epochs: 100)...
Finish Training! (Model is NOT saved)

[0.6618427634239197] 0/66


Dataset 1/66 - Session Exp_2025-05-27/E8331D05289A/
Dataset 2/66 - Session Exp_2025-06-18/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
Start Training (total epochs: 100)...
Finish Training! (Model is NOT saved)

[0.6618427634239197, 0.8506335020065308] 1/66


Dataset 1/66 - Session Exp_2025-05-27/E8331D05289A/
Dataset 2/66 - Session Exp_2025-06-18/E9AD0E7DCC2B/
Dataset 3/66 - Session Exp_2025-06-20-v1/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
Start Training (total epochs: 100)...
Finish Training! (Model is NOT saved)

[0.6618427634239197, 0.8506335020065308, 0.8846801519393921] 2/66


Dataset 1/66 - Session Exp_2025-05-27/E8331D05289A/
Dataset 2/66 - Session Exp_2025-06-18/E9AD0E7DCC2B/
Dataset 3/66 - Session 

In [ ]:
baseline_K_val = [1, 4, 7, 10, 13]

if __name__ == "__main__":
    for K in baseline_K_val:
        trainer = baseline.ModelTrainer(config, subject="Hunmin")
        trainer.run_all_sessions_training_K_data(K)
        df = trainer.save_results(f'../../Results/Results_Hunmin/Baseline_results_train_with_{K}data_H.csv')

# 저장해놨어서 데이터 업데이트 전에 한번 돌리면 굳이 안돌려도 됨'''

In [ ]:
baseline_K_val = [1, 11, 21, 31, 41, 51, 61]

if __name__ == "__main__":
    for K in baseline_K_val:
        trainer = baseline.ModelTrainer(config, subject="Hunmin")
        trainer.run_all_sessions_training_K_data(K)
        df = trainer.save_results(f'../../Results/Results_Hunmin/Baseline_results_train_with_{K}data_H.csv')

# 저장해놨어서 데이터 업데이트 전에 한번 돌리면 굳이 안돌려도 됨'''

In [ ]:
b_path = '//'
#baseline_K = ['1', '11'
#baseline_K = ['1', '4', '7', '10', '13']
baseline_K = ['1', '4', '7', '10', '13', '21', '31', '41', '51', '61']
baselines = []

for idx, K in enumerate(baseline_K):
    baselines.append(b_path + f'/github/Results/Results_Hunmin/Baseline_results_train_with_{K}data_H.csv')

SUBJECT = "Hunmin"

In [ ]:
prog_trainer = Target.ProgressiveTrainer(config, subject=SUBJECT)
Init_acc_all, Prev_acc_all, Trained_acc_all = prog_trainer.run(plot_learning_curve=False)
pd.DataFrame(Trained_acc_all).to_csv(f'../../../Results/Results_Hunmin/Train_Test_per_sessions.csv', index=False)

In [ ]:
SUBJECT = "Hunmin"
prog_trainer = Target.ProgressiveTrainer(config, subject=SUBJECT)
Init_acc_all, Prev_acc_all, Trained_acc_all = prog_trainer.run(plot_learning_curve=False)
pd.DataFrame(Trained_acc_all).to_csv(f'../../../Results/Results_Hunmin/Train_Test_per_sessions_K1_10.csv', index=False)

In [ ]:
for idx, x in enumerate([Init_acc_all, Prev_acc_all, Trained_acc_all]):
    if idx==0:
        print(f"Average of acc without training anything (should be random): {np.mean(x)*100:.2f}%")
    elif idx==1:
        print(f"Average of acc using previous test data after training: {np.mean(x)*100:.2f}%")
    elif idx==2:
        print(f"Average of acc using current test data after training: {np.mean(x)*100:.2f}%")
for idx, b in enumerate(baselines):
    val = float(np.mean(pd.read_csv(baselines[idx])['Accuracy'][int(baseline_K[idx]):]))
    print(f"K-{baseline_K[idx]}: {val:.2f}% | Target: {np.mean(Trained_acc_all[int(baseline_K[idx]):])*100:.2f}")

In [ ]:
for idx, x in enumerate([Init_acc_all, Prev_acc_all, Trained_acc_all]):
    if idx==0:
        print(f"Average of acc without training anything (should be random): {np.mean(x)*100:.2f}%")
    elif idx==1:
        print(f"Average of acc using previous test data after training: {np.mean(x)*100:.2f}%")
    elif idx==2:
        print(f"Average of acc using current test data after training: {np.mean(x)*100:.2f}%")
for idx, b in enumerate(baselines):
    val = float(np.mean(pd.read_csv(baselines[idx])['Accuracy'][int(baseline_K[idx]):]))
    print(f"K-{baseline_K[idx]}: {val:.2f}% | Target: {np.mean(Trained_acc_all[int(baseline_K[idx]):])*100:.2f}")

Average of acc without training anything (should be random): 16.82%
Average of acc using previous test data after training: 77.08%
Average of acc using current test data after training: 90.12%
K-1: 82.19% | Target: 90.12
K-11: 90.64% | Target: 90.59
K-21: 91.87% | Target: 90.30
K-31: 92.00% | Target: 90.13
K-41: 92.50% | Target: 90.39
K-51: 92.43% | Target: 90.41
K-61: 93.75% | Target: 91.49

In [ ]:
b_path = '//'
#baseline_K = ['1', '11'
#baseline_K = ['1', '4', '7', '10', '13']
baseline_K = ['1', '4', '7', '11', '21', '31', '41', '51', '61']
baselines = []

for idx, K in enumerate(baseline_K):
    baselines.append(b_path + f'/github/Results/Results_Hunmin/Baseline_results_train_with_{K}data_H.csv')

SUBJECT = "Hunmin"

In [ ]:
acc_val

In [ ]:
import matplotlib.pyplot as plt

Trained_acc_all_stored = [0.8840176463127136, 0.8607358932495117, 0.9198978543281555, 0.7717928886413574, 0.8730011582374573, 0.938300371170044, 0.8471160531044006, 0.8972570896148682, 0.8645325899124146, 0.8923148512840271, 0.9035354852676392, 0.8530306816101074, 0.9225684404373169, 0.9363487958908081, 0.9387993216514587, 0.9363086819648743, 0.9222248792648315, 0.9043620228767395, 0.9395283460617065, 0.9342013597488403, 0.8649964928627014, 0.8966420292854309, 0.905677855014801, 0.8910960555076599, 0.9078640937805176, 0.9256373643875122, 0.9583430290222168, 0.9348549842834473, 0.9165689945220947, 0.8878569602966309, 0.9041985869407654, 0.8798422813415527, 0.8567448258399963, 0.9363086819648743, 0.9501044154167175, 0.860653817653656, 0.9029744863510132, 0.9052143692970276, 0.8732916116714478, 0.8804171681404114, 0.9026939272880554, 0.9085224866867065, 0.9231662154197693, 0.9049385786056519, 0.9015784859657288, 0.8950173854827881, 0.9117305278778076, 0.8862845301628113, 0.903038740158081, 0.8990273475646973, 0.9011451005935669, 0.9161635041236877, 0.878629207611084, 0.8892243504524231, 0.8921864032745361, 0.8956784605979919, 0.8979781270027161, 0.9057348370552063, 0.9090909361839294, 0.9005793929100037, 0.9035474061965942, 0.926569402217865, 0.9272305965423584, 0.913013219833374, 0.9041985869407654]

# -C: Chair / -C-L: Chair Leg crossed / -F: Floor

X_labels = ['Stand (5/27)', 'Stand (6/15)', 'Sit-C (6/15)', 'Stand (6/20)', 'Sit-C (6/20)',
            'Stand (6/21)', 'Sit-C (6/21)', 'Stand (6/22)', 'Sit-C (6/22)',
            'Stand (6/23)', 'Sit-C (6/23)', 'Stand (6/24)', 'Sit-C (6/24)',
            'Stand (6/26)', 'Sit-C (6/26)', 'Stand (6/26)', 'Sit-C (6/26)',
            'Stand (6/27)', 'Sit-C (6/27)', 'Stand (6/28)', 'Sit-C (6/28)',
            'Stand (6/29)', 'Sit-C (6/29)', 'Stand (6/30)', 'Sit-C (6/30)',
            'Stand (7/1)',  'Sit-C (7/1)',  'Stand (7/2)', 'Sit-C (7/2)',
            'Stand (7/2)',
            'Stand (7/3)',  'Sit-C (7/3)',  'Stand (7/4)', 'Sit-C (7/4)',
            'Stand (7/5)',  'Sit-C (7/5)',  'Stand (7/6)', 'Sit-C (7/6)',
            'Stand (7/7)',  'Sit-C (7/7)',  'Stand (7/8)', 'Sit-C (7/8)',
            'Stand (7/9)',  'Sit-C (7/9)',  'Stand (7/10)', 'Sit-C (7/10)',
            'Stand (7/11)', 'Sit-C (7/11)', 'Stand (7/14)', 'Sit-C (7/14)',
            'Stand (7/17)', 'Sit-C (7/17)', 'Sit-F (7/17)', 'Stand (7/23)', 'Sit-C (7/23)', 'Sit-F (7/23)',
            'Stand (7/24)', 'Sit-C (7/24)', 'Sit-F (7/24)', 'Stand (7/25)', 'Sit-C (7/25)', 'Sit-F (7/25)',
            'Stand (8/1)',  'Sit-C (8/1)', 'Sit-F (8/1)']


plt.rcParams['font.family'] = 'Times New Roman'
x = np.arange(len(X_labels))  # x positions for plotting

plt.figure(figsize=(14, 7), dpi=250)
for idx, base in enumerate(baselines):
    baseline_result = pd.read_csv(base)
    acc_val = baseline_result['Accuracy'].values
    acc_val = acc_val[:len(X_labels)]  # truncate if longer
    plt.plot(x, acc_val / 100, marker='^', label=f'K={baseline_K[idx]}', linestyle='-')

plt.plot(X_labels, Trained_acc_all_stored, c='red', label='Train/Test in same session', marker='o', linestyle='--')

plt.ylim([0, 1])
plt.xlabel('Date (Sessions)', fontsize=18)
plt.ylabel('Test Accuracy', fontsize=18)
plt.xticks(rotation=90, fontsize=10)
plt.legend(fontsize=18, ncol=4)
plt.grid(True, linestyle='--', color='lightgrey')
plt.tight_layout()
plt.show()

In [ ]:
mean_acc

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

n_baselines = len(baselines)
n_sessions = len(X_labels)

bar_width = 0.7
x = np.arange(n_sessions)

plt.figure(figsize=(14, 7), dpi=300)

# Collect all baseline results
all_acc = []
for base in baselines:
    baseline_result = pd.read_csv(base)
    acc_val = baseline_result['Accuracy'].values[:n_sessions]
    acc_val = np.where(acc_val == 0, np.nan, acc_val)
    all_acc.append(acc_val)

all_acc = np.array(all_acc)
bar_width = 0.5

mean_acc = np.nanmean(all_acc, axis=0)
std_acc = np.nanstd(all_acc, axis=0)

mean_acc = np.array([np.nan, 83.4 , 72.42 , 82.24 , 84.65093141, 69.02, 74.165, 81.73518802, 76.48333333, 75.81,
       79.32666667, 78.72790013, 79.3325, 79.9925, 87.1225, 84.44, 86.4525, 87.5075, 80.945, 84.8525,
       85.49, 87.97912719, 87.378, 88.148, 89.224, 88.266, 85.304, 87.908, 89.766, 90.26, 84.464, 86.62882438, 87.86833333, 87.95333333, 89.72333333,
       89.945, 89.27166667, 90.37333333, 91.04666667, 90.66333333, 89.09666667, 90.57629457, 91.58285714, 92.92857143, 92.25,
       92.35857143, 92.03, 92.20714286, 92.47, 93.06857143,92.58714286, 92.85939597, 90.555     , 91.435, 89.395     ,
       88.51, 89.91, 93.12, 91.87125, 92.22375, 87.91, 91.02748295, 90.45333333, 92.97111111, 92.18333333])


plt.bar(x[1:], mean_acc[1:] / 100, width=bar_width, color='skyblue', yerr=std_acc[1:] / 100, capsize=5, ecolor='gray',
        error_kw=dict(lw=1, capsize=3, capthick=1), label='Average of K-session trained models for each session')
#plt.bar(x[1:], mean_acc[1:] / 100, width=bar_width, color='skyblue', yerr=std_acc[1:] / 100, capsize=5, label='Baseline Avg')
#plt.plot(x[1:], mean_acc[1:] / 100, color='blue', label='Average of K-session trained models for each session', marker='o', linestyle='--')
plt.plot(x, Trained_acc_all_stored, c='red', label='Train/Test in same session', marker='o', linestyle='--')


plt.ylim([0.6, 1])
plt.xlabel('Date (Sessions)', fontsize=18)
plt.ylabel('Test Accuracy', fontsize=18)
plt.xticks(x, X_labels, rotation=90, fontsize=10)
plt.legend(fontsize=17, loc='lower right')
plt.grid(True, linestyle='--', color='lightgrey', axis='y')
plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

n_baselines = len(baselines)
n_sessions = len(X_labels)

bar_width = 0.5
x = np.arange(n_sessions)

plt.figure(figsize=(14, 7), dpi=250)

for idx, base in enumerate(baselines):
    baseline_result = pd.read_csv(base)
    acc_val = baseline_result['Accuracy'].values[:n_sessions]

    plt.bar(x, acc_val / 100,
            width=bar_width, alpha=0.5,  # transparency to overlap
            label=f'K={baseline_K[idx]}')

plt.plot(x, Trained_acc_all_stored, c='red',
         label='Train/Test in same session', marker='o', linestyle='--')

#plt.ylim([0, 1])
plt.ylim([0.6, 1])
plt.xlabel('Date (Sessions)', fontsize=18)
plt.ylabel('Test Accuracy', fontsize=18)
plt.xticks(x, X_labels, rotation=90, fontsize=10)
plt.legend(fontsize=14, ncol=3, loc='lower right')
plt.grid(True, linestyle='--', color='lightgrey', axis='y')
plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

n_baselines = len(baselines)
n_sessions = len(X_labels)

bar_width = 0.5
x = np.arange(n_sessions)

plt.figure(figsize=(14, 7), dpi=250)

# Collect all baseline results
all_acc = []
for base in baselines:
    baseline_result = pd.read_csv(base)
    acc_val = baseline_result['Accuracy'].values[:n_sessions]
    all_acc.append(acc_val)

# Convert to array (n_baselines × n_sessions)
all_acc = np.array(all_acc)

# Average accuracy across baselines
mean_acc = all_acc.mean(axis=0)

# Plot one bar per session
plt.bar(x, mean_acc / 100, width=bar_width, color='skyblue', label='Baseline Avg')

plt.plot(x, Trained_acc_all_stored, c='red',
         label='Train/Test in same session', marker='o', linestyle='--')

plt.ylim([0, 1])
plt.xlabel('Date (Sessions)', fontsize=18)
plt.ylabel('Test Accuracy', fontsize=18)
plt.xticks(x, X_labels, rotation=90, fontsize=10)
plt.legend(fontsize=14)
plt.grid(True, linestyle='--', color='lightgrey', axis='y')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 5), dpi=250)
#plt.title('Training from scratch every time', fontsize=15)
plt.plot(Trained_acc_all_stored, c='red', label='Train/Test in same session', marker='o', linestyle='--')
for idx, base in enumerate(baselines):
    baseline_result = pd.read_csv(base)
    plt.plot(X_labels, baseline_result['Accuracy'] / 100, marker='^', label=f'K={baseline_K[idx]}', linestyle='-')

plt.ylim([0.8, 1])
#plt.ylim([0.6, 1])
plt.xlabel('Date (Sessions)', fontsize=18)
plt.ylabel('Test Accuracy', fontsize=18)
plt.xticks(rotation=90, fontsize=10)
plt.legend(fontsize=18, ncol=4)
plt.grid(True, linestyle='--', color='lightgrey')
plt.tight_layout()
plt.show()